# Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, sys
from pathlib import Path
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == 'Multifirefly-Project':
        os.chdir(p)
        sys.path.insert(0, str(p / 'multiff_analysis/multiff_code/methods'))
        break
    
from data_wrangling import specific_utils, process_monkey_information, general_utils, combine_info_utils
from pattern_discovery import pattern_by_trials, pattern_by_trials, cluster_analysis, organize_patterns_and_features
from visualization.matplotlib_tools import plot_behaviors_utils
from neural_data_analysis.neural_analysis_tools.get_neural_data import neural_data_processing
from neural_data_analysis.neural_analysis_tools.visualize_neural_data import plot_neural_data, plot_modeling_result
from neural_data_analysis.neural_analysis_tools.model_neural_data import transform_vars, neural_data_modeling, drop_high_corr_vars, drop_high_vif_vars
from neural_data_analysis.topic_based_neural_analysis.neural_vs_behavioral import prep_monkey_data, prep_target_data, neural_vs_behavioral_class
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural import planning_and_neural_class, pn_utils, pn_helper_class, pn_aligned_by_seg, pn_aligned_by_event
from neural_data_analysis.neural_analysis_tools.cca_methods import cca_class
from neural_data_analysis.neural_analysis_tools.cca_methods import cca_class, cca_utils, cca_cv_utils
from neural_data_analysis.neural_analysis_tools.cca_methods.cca_plotting import cca_plotting, cca_plot_lag_vs_no_lag, cca_plot_cv
from machine_learning.ml_methods import regression_utils, regz_regression_utils, ml_methods_class, classification_utils, ml_plotting_utils, ml_methods_utils
from planning_analysis.show_planning import nxt_ff_utils, show_planning_utils
from neural_data_analysis.neural_analysis_tools.gpfa_methods import elephant_utils, fit_gpfa_utils, plot_gpfa_utils, gpfa_helper_class
from neural_data_analysis.neural_analysis_tools.align_trials import time_resolved_regression, time_resolved_gpfa_regression,plot_time_resolved_regression
from neural_data_analysis.neural_analysis_tools.align_trials import align_trial_utils
from decision_making_analysis.event_detection import detect_rsw_and_rcap

from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_psth import core_stops_psth, psth_postprocessing, psth_stats, compare_events, dpca_utils
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.get_stop_events import get_stops_utils, collect_stop_data

from neural_data_analysis.neural_analysis_tools.glm_tools.glm_fit import general_glm_fit, cv_stop_glm, glm_fit_utils, variance_explained, glm_runner
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_glm.glm_plotting import plot_spikes, plot_glm_fit, plot_tuning_func, compare_glm_fit
from neural_data_analysis.design_kits.design_around_event import event_binning, stop_design, cluster_design, design_checks
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_glm.glm_hyperparams import compare_glm_configs, glm_hyperparams_class
from neural_data_analysis.topic_based_neural_analysis.ff_visibility import ff_vis_epochs, vis_design

from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.get_stop_events import decode_stops_design

# import decoding
from neural_data_analysis.neural_analysis_tools.decoding_tools.event_decoding import decoding_utils, decoding_analysis, plot_decoding, cmp_decode, load_results
from neural_data_analysis.design_kits.design_by_segment import spike_history
from neural_data_analysis.neural_analysis_tools.decoding_tools.general_decoding import cv_decoding
from neural_data_analysis.topic_based_neural_analysis.replicate_one_ff.one_ff_gam import one_ff_gam_fit, gam_variance_explained
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers import multiff_encoding_params
from neural_data_analysis.topic_based_neural_analysis.target_decoder import prep_target_decoder
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers import encoding_design_utils
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers import encoding_design_utils, process_encode_design, encoder_gam_helper

from neural_data_analysis.topic_based_neural_analysis.replicate_one_ff.one_ff_gam import one_ff_gam_fit, one_ff_gam_design, penalty_tuning, backward_elimination, gam_variance_explained, plot_gam_fit
from data_wrangling import combine_info_utils
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_pipelines import base_encoding_task, encoding_models, encoding_runner, encoding_tasks

import sys
import math
import gc
import subprocess
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc
from scipy import linalg, interpolate
from scipy.signal import fftconvolve
from scipy.io import loadmat
from scipy import sparse
import torch
from numpy import pi
import cProfile
import pstats
import json

# Machine Learning imports
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.multivariate.cancorr import CanCorr
import statsmodels.api as sm

# Neuroscience specific imports
import neo
import rcca

# To fit gpfa
import numpy as np
from importlib import reload
from scipy.integrate import odeint
import quantities as pq
import neo
from elephant.spike_train_generation import inhomogeneous_poisson_process
from elephant.gpfa import GPFA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from elephant.gpfa import gpfa_core, gpfa_util

plt.rcParams["animation.html"] = "html5"
os.environ['KMP_DUPLICATE_LIB_OK']='True'
rc('animation', html='jshtml')
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams['animation.embed_limit'] = 2**128
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True)
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

print("done")


%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.max_colwidth', 200)

# new class

## choose task

In [ ]:
task_class = encoding_tasks.StopEncodingTask

In [ ]:
task_class = encoding_tasks.VisEncodingTask

In [ ]:
task_class = encoding_tasks.PNEncodingTask

In [ ]:
task_class = encoding_tasks.FSEncodingTask

## choose model

In [ ]:
# model = encoding_models.RNNModel(cv_mode="blocked_time_buffered")
model = encoding_models.RNNModel(cv_mode="blocked_time_buffered")

In [ ]:
model = encoding_models.PGAMModel(cv_mode="blocked_time_buffered")

# all sessions

## all sessions in same plots

In [ ]:
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers import show_encoding_results

combined_data_exists_ok = False

#monkey_names = ['monkey_Schro']  # or ['monkey_Bruno', 'monkey_Schro']
monkey_names = ['monkey_Bruno']
use_neural_coupling = True

cv_mode = 'blocked_time_buffered'

session_results_list = show_encoding_results.collect_category_ecdf_from_sessions(
    monkey_names,
    raw_data_dir_name='all_monkey_data/raw_monkey_data',
    task_class=task_class,
    bin_width=0.04,
    exists_ok=combined_data_exists_ok,  # load from cache if present; save after computing
    use_neural_coupling=use_neural_coupling,
    cv_mode=cv_mode,
)


In [ ]:
session_results_list[0][1][0]['category_contributions'].keys()

In [ ]:
# Plot: for each category, one ECDF line per session on a single plot
if session_results_list:
    plot_gam_fit.plot_category_ecdf_by_sessions(
        session_results_list,
        metric_key='delta_pseudo_r2_clip_leave',
        log_x=True,
    )

## category_variance

In [ ]:
load_only = True
use_neural_coupling = True

raw_data_dir_name = 'all_monkey_data/raw_monkey_data'
# monkey_names = ['monkey_Bruno', 'monkey_Schro']
monkey_names = ['monkey_Schro']
tuning_feature_mode = 'boxcar_only'
load_if_exists = True

all_session_results = {}  # raw_data_folder_path -> all_neuron_r2

for monkey_name in monkey_names:
    sessions_df = combine_info_utils.make_sessions_df_for_one_monkey(
        raw_data_dir_name, monkey_name
    )
    for _, row in sessions_df.iterrows():
        raw_data_folder_path = os.path.join(
            raw_data_dir_name, row['monkey_name'], row['data_name']
        )
        print('=' * 80)
        print(f'Processing: {raw_data_folder_path}')
        print('=' * 80)
        try:
            prs = multiff_encoding_params.default_prs()
            task = task_class(
                raw_data_folder_path=raw_data_folder_path,
                bin_width=0.04,
                encoder_prs=prs,
            )
            model = encoding_models.PGAMModel(cv_mode='blocked_time_buffered')
            runner = encoding_runner.EncodingRunner(task, model)

            all_results = plot_gam_fit.run_unit_ecdf(runner, use_neural_coupling=use_neural_coupling)

        except Exception as e:
            print(f'[ERROR] Failed for {raw_data_folder_path}: {e}')
            continue

# one session

## instantiate

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0226"
#raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Schro/data_0326"

data_exists_ok = True
use_neural_coupling = True

prs = multiff_encoding_params.default_prs()

task = task_class(
    raw_data_folder_path=raw_data_folder_path,
    bin_width=0.04,
    encoder_prs=prs,
)


runner = encoding_runner.EncodingRunner(task, model, use_neural_coupling=use_neural_coupling)
runner.collect_data(exists_ok=True,
                     )

In [ ]:
runner.meta_df_used

## cv variance explained

### on one neuron

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0220"
#raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Schro/data_0326"

data_exists_ok = True

prs = multiff_encoding_params.default_prs()

task = task_class(
    raw_data_folder_path=raw_data_folder_path,
    bin_width=0.04,
    encoder_prs=prs,
)

In [ ]:

data_exists_ok = True

prs = multiff_encoding_params.default_prs()

runner.collect_data(exists_ok=data_exists_ok,
                     )

# do crossval on one neuron
unit_idx = 5
# After runner.collect_data(exists_ok=True)
runner.get_gam_save_paths(unit_idx=unit_idx)

# Cross-validated variance explained
cv_res = runner.crossval(
    unit_idx=unit_idx,
    n_folds=5,
    fit_kwargs=dict(l1_groups=[], max_iter=1000, tol=1e-6, verbose=False, save_path=None),
    load_if_exists=True,
    buffer_samples=20,
)
print(cv_res['mean_classical_r2'], cv_res['mean_pseudo_r2'])

### iterate through neurons

In [ ]:
data_exists_ok = True

prs = multiff_encoding_params.default_prs()

runner.collect_data(exists_ok=data_exists_ok,
                     )

In [ ]:
all_neuron_r2 = runner.crossval_all_neurons(
    n_folds=5,
    load_if_exists=True,
    buffer_samples=20,
    verbose=True,
    plot_cdf=True,
    log_x=False,
)

In [ ]:
print('all_neuron_r2', all_neuron_r2)
if all_neuron_r2 is not None:
    all_session_results[raw_data_folder_path] = all_neuron_r2
    gam_variance_explained.plot_variance_explained_cdf(all_neuron_r2, log_x=False)
    

### fold diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Reload per-fold R² from cached results (no recomputation)
fold_records = []
for unit_idx in range(len(all_neuron_r2)):
    res = runner.crossval(
        unit_idx,
        n_folds=5,
        load_if_exists=True,
        buffer_samples=20,
        verbose=False,
    )
    for fold_i, (r2_cl, r2_ps) in enumerate(
        zip(
            res.get('fold_classical_r2', res.get('all_folds_classical_r2', [])),
            res.get('fold_pseudo_r2',    res.get('all_folds_pseudo_r2',    [])),
        )
    ):
        fold_records.append({
            'unit_idx': unit_idx,
            'fold': fold_i,
            'classical_r2': float(r2_cl),
            'pseudo_r2': float(r2_ps),
        })

fold_df = pd.DataFrame(fold_records)

# Heatmap: neurons × folds, sorted by mean pseudo R²
pivot = fold_df.pivot(index='unit_idx', columns='fold', values='pseudo_r2')
pivot_sorted = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(6, max(4, len(all_neuron_r2) * 0.3)))
im = ax.imshow(pivot_sorted.values, aspect='auto', vmin=-0.2, vmax=0.2, cmap='RdBu_r')
ax.set_xlabel('Fold')
ax.set_ylabel('Neuron (sorted by mean pseudo R²)')
ax.set_xticks(range(pivot_sorted.shape[1]))
ax.set_title('Per-fold pseudo R² (encoding)')
plt.colorbar(im, ax=ax, label='Pseudo R²')
plt.tight_layout()
plt.show()


In [ ]:
stop!

## max dependency

In [ ]:
runner.get_max_temporal_dependency()

## category variance

### all neurons

In [ ]:
all_results = plot_gam_fit.run_unit_ecdf(runner)


## anova / lm

ANOVA and LM are encoding questions (behavioral variable → neural response).
They operate on `task.raw_behavioral_feats` — identical to `feats_to_decode`
from the corresponding decoding task.

In [ ]:
# ── 1. Discover categorical variables ───────────────────────────────
cat_vars = runner.find_categorical_vars()
print('categorical vars:', cat_vars)

# ── 2. Run ANOVA (each variable tested independently) ────────────────
anova = runner.run_anova_all_neurons(alpha=0.05)

# ── 3. Run LM (partial F-test, controls for all other variables) ─────
#    include_all_feats=True folds raw_behavioral_feats into the model
#    as unreported covariates (after VIF/correlation reduction).
#    Cache is saved automatically next to encoding_design/.
lm = runner.run_lm_all_neurons(include_all_feats=True, alpha=0.05)

# ── 4. Compare ───────────────────────────────────────────────────────
fig = runner.plot_lm_vs_anova_results(anova, lm)
plt.show()

In [ ]:
runner.raw_behavioral_feats.head()

## penalty tuning

Here, the cv_score in run_penalty_tuning is the mean held-out Poisson log-likelihood from cross-validation. The best lambdas are those with the highest mean held-out log-likelihood.

In [ ]:
# unit_idx = 0
# tuning = runner.run_penalty_tuning(
#     unit_idx,
#     n_folds=5,
#     group_name_map=None,  # default groups; or map lam keys -> group names
#     load_if_exists=True,
# )
# # tuning['best_lams'], tuning['cv_results'], tuning['save_path']

# df, df_sorted = penalty_tuning.analyze_penalty_tuning(tuning)

In [ ]:
# for var_name in ['lam_f', 'lam_g', 'lam_h']:
#     penalty_tuning.plot_penalty_tuning_for_one_var(runner, var_name)

## backward elimination

In [ ]:
unit_idx = 1

elim_res = runner.run_backward_elimination(
    unit_idx,
    alpha=0.05,
    n_folds=10,
    load_if_exists=True,
)


print([g.name for g in elim_res["kept_groups"]])
print(elim_res["history_csv"])

print('kept_groups', elim_res["kept_groups"])

eliminated = [h["removed"] for h in elim_res["history"]]
print('eliminated', eliminated)

## cv tuning curves

In [ ]:
for unit_idx in range(runner.num_neurons):
    groups = runner.get_gam_groups()
    fold_coef_results = runner.crossval_tuning_curve_coef(unit_idx)
    plot_gam_fit.plot_fold_coefficient_by_variable(fold_coef_results, runner.structured_meta_groups, var_list=None, use_sem=True)

## fraction tuned

In [ ]:
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers.encoding_model_utils import (
    calculate_tuning_fraction_from_backward_elimination,
    plot_fraction_tuned,
)

unit_indices = list(range(runner.task.binned_spikes.shape[1]))

elim_results = []
for unit_idx in unit_indices:
    res = runner.run_backward_elimination(
        unit_idx=unit_idx,
        load_if_exists=True,
        n_folds=10,
        alpha=0.05,
    )
    elim_results.append(res)

tuning_stats = calculate_tuning_fraction_from_backward_elimination(elim_results)
fig, ax = plot_fraction_tuned(tuning_stats)

# RNN encoding model

GRU-based Poisson encoding model. By default (`use_raw_feats=True`) the
input is `task.raw_behavioral_feats` — raw kinematics with no splines, exact
parity with `feats_to_decode` from the corresponding decoding task.
Set `use_raw_feats=False` to feed the full spline-expanded `binned_feats`.

### instantiate RNN runner

In [ ]:
# Swap in RNNModel — task data (already collected) is reused as-is.
rnn_model = encoding_models.RNNModel(
    hidden_dim=64,
    n_epochs=100,
    lr=3e-4,
    cv_mode="blocked_time_buffered",
    buffer_samples=20,
    use_raw_feats=True,   # raw kinematics — matches feats_to_decode
    # use_raw_feats=False,  # spline-expanded binned_feats instead
)
rnn_runner = encoding_runner.EncodingRunner(task, rnn_model)
# task data is already loaded; no need to re-run collect_data
rnn_runner.collect_data(exists_ok=True)

### inspect input features

In [ ]:
# Confirm shape of what the RNN actually sees
X = rnn_runner.raw_behavioral_feats
print('RNN input shape:', X.shape)
print('RNN input columns:', X.columns.tolist())

### one neuron

In [ ]:
unit_idx = 5

cv_res = rnn_runner.crossval(
    unit_idx=unit_idx,
    n_folds=5,
    load_if_exists=True,
)
print(f'unit {unit_idx}  R² train={cv_res["mean_r2_train"]:.3f}  '
      f'R² test={cv_res["mean_r2_test"]:.3f}')

### all neurons

In [ ]:
rnn_all = rnn_runner.crossval_all_neurons(
    n_folds=5,
    load_if_exists=True,
    verbose=True,
)

### compare raw_feats vs spline_feats

In [ ]:
# Optionally run a second RNN on spline-expanded features to compare.
rnn_model_spline = encoding_models.RNNModel(
    hidden_dim=64,
    n_epochs=100,
    lr=3e-4,
    cv_mode="blocked_time_buffered",
    use_raw_feats=False,  # spline-expanded binned_feats
    rnn_results_subdir="rnn_results_spline",  # separate cache
)
rnn_runner_spline = encoding_runner.EncodingRunner(task, rnn_model_spline)
rnn_runner_spline.collect_data(exists_ok=True)

rnn_all_spline = rnn_runner_spline.crossval_all_neurons(
    n_folds=5,
    load_if_exists=True,
    verbose=True,
)

# exp (reduce design)

In [ ]:
stop!

In [ ]:
#raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0220"
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0316"

data_exists_ok = True

prs = multiff_encoding_params.default_prs()

task = task_class(
    raw_data_folder_path=raw_data_folder_path,
    bin_width=0.04,
    encoder_prs=prs,
)
model = encoding_models.PGAMModel(cv_mode="blocked_time_buffered")
runner = encoding_runner.EncodingRunner(task, model)
runner.collect_data(exists_ok=data_exists_ok,
                     )

design_df, _ = spike_history.add_spike_history_to_design(
    design_df=runner.binned_feats.reset_index(drop=True),
    colnames=runner.spk_colnames,
    X_hist=runner.X_hist,
    target_col=target_col,
    include_runner=True,
    cross_neurons=cross_neurons,
    meta_groups=None,
)

In [ ]:
binned_feats_reduced = process_encode_design.reduce_encoding_design(runner.binned_feats, corr_threshold_for_lags=0.98, 
                         vif_threshold_for_initial_subset=20, 
                         vif_threshold=20, 
                         verbose=True)

In [ ]:
runner.get_design_for_unit(5)
final_df_reduced = process_encode_design.reduce_encoding_design(runner.design_df, corr_threshold_for_lags=0.98, 
                         vif_threshold_for_initial_subset=20, 
                         vif_threshold=20, 
                         verbose=True)

In [ ]:
strong_corr_df = process_encode_design.get_strong_correlations(df_reduced, threshold=0.7)
print(strong_corr_df)

In [ ]:
stop!

# exp (encode_vis)

In [ ]:
import os

# Neuroscience specific imports
from neural_data_analysis.design_kits.design_by_segment import spike_history
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_helpers import encoding_design_utils, process_encode_design, encoder_gam_helper
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.get_stop_events import (
    collect_stop_data
)
from neural_data_analysis.topic_based_neural_analysis.ff_visibility import decode_vis_utils



In [ ]:
#raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0220"
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0316"

data_exists_ok = False

prs = multiff_encoding_params.default_prs()

task = task_class(
    raw_data_folder_path=raw_data_folder_path,
    bin_width=0.04,
    encoder_prs=prs,
)
model = encoding_models.PGAMModel(cv_mode="blocked_time_buffered")
runner = encoding_runner.EncodingRunner(task, model)
# runner.collect_data(exists_ok=data_exists_ok,
#                      )

In [ ]:

# runner.pn = collect_stop_data.init_pn_to_collect_stop_data(
#     runner.raw_data_folder_path, bin_width=0.04)
runner.pn, datasets, _ = collect_stop_data.collect_stop_data_func(
    runner.raw_data_folder_path,
    bin_width=runner.bin_width,
)
runner.pn.make_or_retrieve_ff_dataframe()


In [ ]:

new_seg_info, events_with_stats = decode_vis_utils.prepare_new_seg_info(
    runner.pn.ff_dataframe,
    runner.pn.bin_width,
)

In [ ]:
new_seg_info

In [ ]:


print('[EncodingRunner] Computing design matrices from scratch')
design_kwargs = runner._encoding_design_kwargs()


runner.ff_on_df, runner.group_on_df = decode_vis_utils.extract_ff_visibility_tables_fast(runner.pn.ff_dataframe)



(
    runner.pn,
    runner.binned_spikes,
    runner.binned_feats,
    runner.meta_df_used,
    runner.temporal_meta,
    runner.tuning_meta,
) = encoding_design_utils.build_vis_encoding_design(
    runner.pn,
    datasets,
    new_seg_info,
    events_with_stats,
    runner.ff_on_df,
    runner.group_on_df,
    runner.bin_width,
    add_stop_cluster_features=False,
    add_retry_features=False,
    **design_kwargs,
)

In [ ]:
new_seg_info

In [ ]:
runner.meta_df_used

In [ ]:
runner.bin_df = spike_history.make_bin_df_from_meta_df(runner.meta_df_used)


In [ ]:
runner.binned_feats

In [ ]:
runner._prepare_spike_history_components()


In [ ]:
runner.bin_df = spike_history.make_bin_df_from_meta_df(runner.meta_df_used)

runner._prepare_spike_history_components()

runner._make_structured_meta_groups()

runner._save_design_matrices()

In [ ]:
runner.get_design_for_unit(0)
runner.design_df.head()

# Appendix

In [ ]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
runner.binned_feats.shape

In [ ]:
runner.binned_feats.describe().T

In [ ]:
runner.binned_feats.describe().T.sort_values(by='std', ascending=False)

## how long does it take to make X_hist

In [ ]:

if not hasattr(runner.pn, "spikes_df"):
    runner.pn = collect_stop_data.init_pn_to_collect_stop_data(
        runner.raw_data_folder_path, bin_width=0.04
    )

n_basis_hist = getattr(runner.encoder_prs, "default_n_basis", 20)
spike_hist_t_max = runner.binrange_dict["spike_hist"][1]

_, runner._basis, runner.spk_colnames, _, runner.X_hist = (
    spike_history.build_design_with_spike_history_from_bins(
        spikes_df=runner.pn.spikes_df,
        bin_df=runner.bin_df,
        X_pruned=runner.binned_feats,
        meta_groups={},
        dt=runner.bin_width,
        t_max=spike_hist_t_max,
        returnX_hist=True,
        n_basis=n_basis_hist,
    )
)

## check cluster info

In [ ]:
df = runner.meta_df_used
df

In [ ]:
# Meta columns that contain cluster info
meta_cluster_cols = [c for c in runner.meta_df_used.columns if 'cluster' in c.lower() or c == 'is_clustered']
runner.meta_df_used[['event_id', 'k_within_seg'] + meta_cluster_cols]

In [ ]:
# Counts
runner.binned_feats['is_clustered'].value_counts()
runner.binned_feats['event_cluster_id'].nunique()

# Rows with no cluster (event_cluster_id NaN)
runner.binned_feats['event_cluster_id'].isna().sum()

# Cluster size distribution (clustered events only)
runner.binned_feats.loc[runner.binned_feats['is_clustered'] == 1, 'event_cluster_size'].value_counts()

In [ ]:
runner.get_design_for_unit(0)  # NaN columns already filled with 0
df = runner.design_df.copy()
cluster_cols = [c for c in df.columns if 'cluster' in c.lower() or c == 'is_clustered']
df[cluster_cols].describe()

## debug stop_category_df

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0221"
bin_width = 0.04


In [ ]:
runner.pn.stop_category_df

In [ ]:
pn, datasets, new_seg_info, events_with_stats = \
    decode_stops_design.prepare_stop_design_inputs(
        raw_data_folder_path, bin_width)

In [ ]:
pn.stop_category_df

In [ ]:
unique_stops_df = get_stops_utils.extract_unique_stops(pn.monkey_information)
unique_stops_df

## debug job script

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0219"

In [ ]:

prs = multiff_encoding_params.default_prs()

task = encoding_tasks.StopEncodingTask(
    raw_data_folder_path=raw_data_folder_path,
    bin_width=0.04,
    encoder_prs=prs,
)
model = encoding_models.PGAMModel(cv_mode="blocked_time_buffered")
runner = encoding_runner.EncodingRunner(task, model)

runner.collect_data(
    exists_ok=True,
)

lambda_config = {
    "lam_f": 100.0,
    "lam_g": 10.0,
    "lam_h": 10.0,
    "lam_p": 10.0,
}

all_neuron_r2 = runner.crossval_all_neurons(
    buffer_samples=20,
    verbose=True,
    plot_cdf=False,
    log_x=False,
)


runner.run_full_analysis_all_neurons()